# 31 — Special Tokens, Padding, Controls, and Prompt Formats

**Master syllabus coverage:** V2 4.8–4.9, 4.13

## Why this matters

BOS/EOS, roles, and style controls define a machine-readable protocol. If serialization or loss masking differs between training and inference, model behavior fails even when architecture and weights are correct.

## Learning objectives

- Assign noncolliding stable IDs to protocol tokens.
- Serialize a structured prompt deterministically.
- Align input IDs, shifted targets, and assistant-only loss masks.
- Validate malformed formats and literal-text collision policies.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Special tokens are protocol symbols

Common roles include BOS (start), EOS (stop), PAD (batch alignment), UNK (only when
coverage is incomplete), separators, roles, and task/control tokens. Each must have one
stable ID, documented insertion rules, and training examples that teach its meaning.
Reserve them before ordinary vocabulary construction so IDs cannot collide.


In [ ]:
SPECIALS = ["<PAD>", "<BOS>", "<EOS>", "<USER>", "<ASSISTANT>",
            "<STYLE_DRY>", "<STYLE_PUN>"]
ordinary = ["tell", "me", "a", "joke", "timing", "matters", "."]
vocabulary = [*SPECIALS, *ordinary]
stoi = {token: index for index, token in enumerate(vocabulary)}
itos = dict(enumerate(vocabulary))
assert len(stoi) == len(vocabulary)
print(stoi)


## 2. Prompt formatting is part of the training distribution

The model sees tokens, not abstract chat messages. Training and inference must serialize
roles, controls, separators, and EOS identically. A format should be unambiguous even when
user text contains strings that visually resemble control tokens.


In [ ]:
def format_example(user_tokens: list[str], answer_tokens: list[str], style: str) -> list[str]:
    style_token = {"dry": "<STYLE_DRY>", "pun": "<STYLE_PUN>"}[style]
    return ["<BOS>", style_token, "<USER>", *user_tokens,
            "<ASSISTANT>", *answer_tokens, "<EOS>"]

example = format_example(["tell", "me", "a", "joke"], ["timing", "matters", "."], "dry")
ids = [stoi[token] for token in example]
decoded = [itos[index] for index in ids]
assert decoded == example and ids[-1] == stoi["<EOS>"]
print(example)
print(ids)


## 3. Loss masks decide which tokens teach the model

In instruction tuning, the input prompt often supplies context but only assistant tokens
contribute to the supervised loss. A boolean loss mask must align exactly with shifted
targets. Pretraining generally predicts every non-padding token; do not reuse one masking
policy blindly across objectives.


In [ ]:
assistant_position = example.index("<ASSISTANT>")
# Target at position t is example[t+1]. Include tokens after the assistant marker.
inputs = ids[:-1]
targets = ids[1:]
loss_mask = [position >= assistant_position for position in range(len(targets))]
supervised_targets = [itos[target] for target, include in zip(targets, loss_mask) if include]
print("supervised targets:", supervised_targets)
assert supervised_targets == ["timing", "matters", ".", "<EOS>"]


## 4. Literal-text collisions and escaping

If raw user text can become the same ID as `<ASSISTANT>`, it can alter structure. Robust
systems tokenize structured messages through an API rather than string concatenation,
escape reserved sequences, or use special IDs that ordinary text encoding cannot emit.
Test malformed role order, missing EOS, duplicate BOS, and controls absent from training.


In [ ]:
def validate_protocol(tokens: list[str]) -> None:
    assert tokens[0] == "<BOS>" and tokens[-1] == "<EOS>"
    assert tokens.count("<BOS>") == 1 and tokens.count("<EOS>") == 1
    assert tokens.count("<USER>") == 1 and tokens.count("<ASSISTANT>") == 1
    assert tokens.index("<USER>") < tokens.index("<ASSISTANT>")

validate_protocol(example)
malformed = ["<BOS>", "<ASSISTANT>", "joke", "<USER>", "me", "<EOS>"]
try:
    validate_protocol(malformed)
except AssertionError:
    print("malformed role order correctly rejected")


## Failure modes to recognize

- **Special/ordinary ID collision:** decoded meaning depends on code path rather than the ID.
- **Train/serve format drift:** inference context is out of distribution.
- **Off-by-one loss mask:** prompt or marker tokens receive unintended supervision.
- **Untrained control:** a reserved token is expected to steer behavior without relevant examples.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Design a multi-turn prompt grammar and write validators for legal role order.
2. Create input, target, attention, and loss masks for two padded instruction examples.
3. Specify how ordinary user text containing `<EOS>` should encode without ending the sequence.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can define, serialize, mask, round-trip, and validate the complete token protocol used by both training and inference.

**Next:** 32 — Context windows, masks, packing, and storage.
